# Chapter 3: Feature Engineering

## Your Model Can't Read English — Here's How to Translate

At this point in the pipeline, we have validated data with standardized column names. But we're not ready for a model yet.

A model is just math. It multiplies, adds, and compares numbers. It cannot:
- Understand that `contract_type = "month-to-month"` means high risk
- Handle a blank cell (NaN) — it crashes or produces garbage
- Fairly compare `tenure_months` (range: 1-72) with `monthly_charges` (range: 18-118)

Feature engineering is the **prep chef** — it takes raw, validated ingredients and chops them into a form the model can digest. Four steps:

| Step | What It Does | Why |
|------|-------------|-----|
| Imputation | Fills blank cells | Model crashes on NaN |
| Encoding | Converts categories to numbers | Model only speaks numbers |
| Interaction features | Creates meaningful combinations | Captures relationships |
| Scaling | Puts everything on same range [0,1] | Prevents unfair dominance |

In [ ]:
import pandas as pd
import numpy as np
from churn_pipeline.steps.feature_engineering import engineer_features, FeatureArtifacts

## Step 0: Start With Validated Data

This is what comes out of Chapter 2 — data that passed the bouncer. Column names are standardized, types are (mostly) correct, but there are still blanks, strings, and uneven scales.

In [ ]:
# Realistic validated data — notice the problems a model can't handle
validated_data = pd.DataFrame({
    "customer_id": ["C001", "C002", "C003", "C004", "C005", "C006", "C007"],
    "tenure_months": [1, 34, 2, 45, None, 12, 60],        # NaN! Model will crash
    "monthly_charges": [29.85, 56.95, 53.85, 42.30, 89.10, None, 105.45],  # NaN!
    "total_charges": [29.85, 1889.5, 108.15, 1840.75, 2000.0, 500.0, 6327.0],
    "churn_label": [0, 1, 1, 0, 1, 0, 0],
    "contract_type": ["month-to-month", "one_year", "month-to-month", "two_year",
                       "month-to-month", None, "two_year"],  # String! And NaN!
    "support_tickets": [0, 3, 5, 0, 2, 1, 0],
    "payment_method": ["electronic_check", "credit_card", "electronic_check",
                        "bank_transfer", "credit_card", "electronic_check", "bank_transfer"],
})

print("Validated data (still has problems for a model):")
print(validated_data)
print(f"\nNull values per column:")
print(validated_data.isnull().sum())
print(f"\nColumn types:")
print(validated_data.dtypes)

## The Transformation: Before and After

One function call takes us from messy validated data to a clean numeric matrix. Let's see the full transformation.

In [ ]:
# Training mode: learn the transformation rules AND apply them
feature_matrix, artifacts = engineer_features(validated_data, fit=True)

print("AFTER feature engineering:")
print(f"  Shape: {feature_matrix.shape} (7 customers × {feature_matrix.shape[1]} features)")
print(f"  Data type: {feature_matrix.dtype}")
print(f"  Any NaN values: {np.isnan(feature_matrix).any()}")
print(f"  Value range: [{feature_matrix.min():.3f}, {feature_matrix.max():.3f}]")
print(f"\nFeature names (in order):")
for i, name in enumerate(artifacts.feature_names):
    print(f"  {i}: {name}")

In [ ]:
# Let's look at the actual numbers
feature_df = pd.DataFrame(feature_matrix, columns=artifacts.feature_names)
print("The feature matrix (what the model actually sees):")
print(feature_df.round(3))

## Unpacking Each Step

### Step 1: Imputation — Filling the Blanks

Missing values are poison for models. We fill them with sensible defaults:
- **Numeric columns** → median (the middle value, robust to outliers)
- **Categorical columns** → mode (the most common value)

Why median instead of mean? If one customer has `$10,000` monthly charges (probably a data error), the mean gets dragged up, but the median barely moves.

In [ ]:
# What imputation values were learned?
print("Learned imputation values (what fills the blanks):")
print("=" * 50)
for field, value in artifacts.impute_values.items():
    print(f"  {field:20s} → {value}")

print("\nFor example:")
print(f"  tenure_months had a NaN → filled with median = {artifacts.impute_values.get('tenure_months')}")
print(f"  monthly_charges had a NaN → filled with median = {artifacts.impute_values.get('monthly_charges')}")
print(f"  contract_type had a NaN → filled with mode = '{artifacts.impute_values.get('contract_type')}'")

### Step 2: Encoding — Categories Become Numbers

A model can't multiply by `"month-to-month"`. Each unique category gets an integer:

```
month-to-month → 0
one_year       → 1  
two_year       → 2
```

The encoder remembers this mapping forever. When scoring data arrives months later with `"one_year"`, it still gets the same number `1`.

In [ ]:
# What encodings were learned?
print("Learned category encodings:")
print("=" * 50)
for col, encoder in artifacts.encoders.items():
    print(f"\n  {col}:")
    for i, cls in enumerate(encoder.classes_):
        print(f"    '{cls}' → {i}")

### Step 3: Interaction Features — Combinations That Tell a Story

Some relationships are invisible when looking at features alone:

- Customer A: monthly_charges=100, tenure=2 months → paid $200 total
- Customer B: monthly_charges=100, tenure=36 months → paid $3,600 total

Same monthly charges. Completely different relationships with the company. The interaction feature `monthly_charges × tenure_months` captures "approximate lifetime value" — something neither feature alone conveys.

In [ ]:
# Show the interaction feature alongside its components
print("Interaction feature: monthly_charges × tenure_months")
print("=" * 55)
print(f"{'Customer':<10} {'Charges':<10} {'Tenure':<10} {'Interaction (raw)'}")
print("-" * 55)

for i in range(len(validated_data)):
    charges = validated_data['monthly_charges'].iloc[i]
    tenure = validated_data['tenure_months'].iloc[i]
    # After imputation, the interaction would be computed from filled values
    charges_filled = charges if not pd.isna(charges) else artifacts.impute_values.get('monthly_charges', 0)
    tenure_filled = tenure if not pd.isna(tenure) else artifacts.impute_values.get('tenure_months', 0)
    interaction = charges_filled * tenure_filled
    print(f"  C{i+1:<7} ${charges_filled:<9.2f} {tenure_filled:<10.0f} {interaction:.2f}")

### Step 4: Scaling — Leveling the Playing Field

After imputation and encoding, we have all numbers — but they're on wildly different scales:
- `tenure_months`: 1 to 72
- `monthly_charges`: 18 to 118
- `total_charges`: 29 to 8,000+
- `support_tickets`: 0 to 10

A model might over-weight total_charges simply because the numbers are bigger. Min-Max scaling squishes everything to [0, 1]:

```
scaled_value = (value - min) / (max - min)
```

Now 0.5 means "halfway between lowest and highest" regardless of the original unit.

In [ ]:
# Show the scaling ranges learned
print("Scaling ranges (learned from training data):")
print("=" * 60)
print(f"{'Feature':<30} {'Min (→0.0)':<15} {'Max (→1.0)'}")
print("-" * 60)
for i, name in enumerate(artifacts.feature_names):
    min_val = artifacts.scaler.data_min_[i]
    max_val = artifacts.scaler.data_max_[i]
    print(f"  {name:<28} {min_val:<15.2f} {max_val:.2f}")

## Training Mode vs. Scoring Mode

This is the most critical concept in feature engineering:

- **Training mode** (`fit=True`): Look at the data, learn the rules ("median tenure is 29", "there are 3 contract types", "charges range 18-118"). Apply those rules.

- **Scoring mode** (`fit=False`): DON'T look at the new data to learn anything. Just apply the old rules from training.

**Why does this matter?** Imagine during training, tenure ranged 1-72. Months later, scoring data arrives with a customer at tenure=84 (longer than anyone in training). If we re-fit the scaler, `0.5` now means something different than what the model learned. The model would see a "different language" than it was trained on.

Scoring mode prevents this by ALWAYS using the training-time rules.

In [ ]:
# New data arrives for scoring (different customers, different values)
scoring_data = pd.DataFrame({
    "customer_id": ["NEW_001", "NEW_002"],
    "tenure_months": [84, 6],           # 84 is beyond training range (1-60)!
    "monthly_charges": [120.0, 25.0],   # 120 is beyond training range!
    "total_charges": [10080.0, 150.0],
    "churn_label": [0, 1],
    "contract_type": ["two_year", "month-to-month"],
    "support_tickets": [0, 7],
    "payment_method": ["bank_transfer", "electronic_check"],
})

# Score using TRAINING-TIME artifacts (not re-fitting!)
scored_matrix, _ = engineer_features(scoring_data, fit=False, artifacts=artifacts)

print("Scoring new data with training-time rules:")
print(f"  Shape: {scored_matrix.shape}")
print(f"  Any NaN: {np.isnan(scored_matrix).any()}")
print(f"  Range: [{scored_matrix.min():.3f}, {scored_matrix.max():.3f}]")
print(f"\n  Notice: values are clipped to [0, 1] even though some exceeded")
print(f"  the training range. The model never sees out-of-range values.")

scored_df = pd.DataFrame(scored_matrix, columns=artifacts.feature_names)
print(f"\n{scored_df.round(3)}")

## Idempotence: Scoring Mode is a Pure Function

A critical property: running scoring mode twice with the same artifacts produces **bit-for-bit identical** results. No randomness, no state changes. Same input + same artifacts = same output, always.

If this property breaks, your predictions become non-reproducible — a customer's risk score could change between two runs even though nothing about them changed.

In [ ]:
# Run scoring mode twice — results must be identical
result_1, _ = engineer_features(scoring_data, fit=False, artifacts=artifacts)
result_2, _ = engineer_features(scoring_data, fit=False, artifacts=artifacts)

identical = np.array_equal(result_1, result_2)
print(f"Run 1 == Run 2: {identical}")
print(f"Max difference: {np.abs(result_1 - result_2).max()}")

if identical:
    print("\n✓ Scoring mode is idempotent — same input always gives same output")

## The Three Guarantees

No matter what valid data goes in, the feature engineering module guarantees:

1. **Same row count** — no customers disappear or duplicate
2. **All float64** — every cell is a number, ready for math
3. **Zero NaN values** — no blanks that would crash the model

We verify these with property-based testing (Hypothesis) — 200 random DataFrames, each with different sizes, null patterns, and column subsets.

In [ ]:
# Quick demonstration with random data
print("Testing the three guarantees with random data:")
print("=" * 50)

for trial in range(5):
    n = np.random.randint(5, 100)
    random_df = pd.DataFrame({
        "customer_id": [f"R{i}" for i in range(n)],
        "tenure_months": np.random.choice([*np.random.randint(1, 72, n-2).tolist(), None, None]),
        "monthly_charges": np.random.uniform(18, 118, n).tolist(),
        "total_charges": np.random.uniform(100, 8000, n).tolist(),
        "churn_label": np.random.randint(0, 2, n).tolist(),
    })
    # Inject some random nulls
    for col in ["monthly_charges", "total_charges"]:
        mask = np.random.random(n) < 0.1
        random_df.loc[mask, col] = None
    
    matrix, _ = engineer_features(random_df, fit=True)
    
    row_check = matrix.shape[0] == n
    dtype_check = matrix.dtype == np.float64
    nan_check = not np.isnan(matrix).any()
    
    status = "✓" if all([row_check, dtype_check, nan_check]) else "✗"
    print(f"  Trial {trial+1}: {n} rows, nulls injected → "
          f"rows={row_check}, float64={dtype_check}, no_nan={nan_check} {status}")

print("\nAll guarantees hold!")

## The Artifacts: What Gets Saved

When training completes, the `FeatureArtifacts` object is saved to S3. It contains everything needed to reproduce the exact same transformation on future data:

- **Scaler** — the min/max values per feature
- **Encoders** — the category→number mapping per column
- **Impute values** — what to fill blanks with
- **Feature names** — the exact column order the model expects

Without these artifacts, scoring would be impossible — you'd have to retrain just to transform new data.

In [ ]:
# Summary of what's stored in artifacts
print("FeatureArtifacts contents:")
print("=" * 50)
print(f"\n1. Scaler: MinMaxScaler fitted on {len(artifacts.feature_names)} features")
print(f"2. Encoders: {len(artifacts.encoders)} categorical columns encoded")
for col in artifacts.encoders:
    print(f"     - {col}: {len(artifacts.encoders[col].classes_)} categories")
print(f"3. Impute values: {len(artifacts.impute_values)} columns with fill values")
print(f"4. Feature names: {artifacts.feature_names}")
print(f"\nThis object is serialized (pickle) and stored alongside the model.")
print(f"Scoring jobs load it to ensure identical transformations.")

## Key Takeaways

1. **Models only speak numbers** — feature engineering is the translation layer
2. **Four steps:** impute → encode → interact → scale
3. **Training mode learns rules, scoring mode applies them** — never re-learn during scoring
4. **Three guarantees:** same row count, all float64, zero NaN
5. **Artifacts are essential** — without them, you can't reproduce the transformation
6. **Idempotence in scoring mode** — same input always gives same output

Next: Chapter 4 shows how we feed this numeric matrix to XGBoost — a team of 100 simple decision-makers that learn from each other's mistakes.

---

*Source code: `src/churn_pipeline/steps/feature_engineering.py`*  
*Tests: `tests/property/test_feature_engineering.py`*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*